In [ ]:
# One-cell Colab: enable a T4 GPU, then run this cell.
import codecs, os, selectors, shutil, subprocess, sys, time
from pathlib import Path
from google.colab import drive

BRANCH = 'feature/hada-attentive-statistics-pooling'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
REPO_PATH = Path('/content/tcformer-full-mamba')
USE_RA = False       # True: enable Riemannian Alignment preprocessing
USE_IM_TTA = False   # True: enable source-free target adaptation after training
IM_TTA_STEPS = 5
if USE_IM_TTA and IM_TTA_STEPS < 1:
    raise ValueError('IM_TTA_STEPS must be positive when USE_IM_TTA=True')
RUN_TAG = f'ra-{int(USE_RA)}_imtta-{IM_TTA_STEPS if USE_IM_TTA else 0}'

if Path('/content/drive/MyDrive').is_dir():
    DRIVE_ROOT = Path('/content/drive')
else:
    mountpoint = Path('/content/drive')
    if mountpoint.exists() and any(mountpoint.iterdir()):
        mountpoint = Path('/content/google_drive')
    drive.mount(str(mountpoint))
    DRIVE_ROOT = mountpoint

MNE_DATA = DRIVE_ROOT / 'MyDrive/datasets/PhysioNetMI_MNE'
RESULT_ARCHIVE = DRIVE_ROOT / f'MyDrive/TCFormer-results/full_mamba_physionet20_4p1s_bs96_ep125_{RUN_TAG}_results'
TRAIN_LOG = DRIVE_ROOT / f'MyDrive/TCFormer-results/full_mamba_physionet20_4p1s_bs96_ep125_{RUN_TAG}.log'

def run(command, cwd=None, env=None, stream=False, log_path=None):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    if not stream:
        subprocess.run(command, cwd=str(cwd) if cwd else None, env=env, check=True)
        return
    # Forward raw chunks instead of lines: tracebacks, tqdm carriage returns,
    # and partial messages are all rendered immediately in the Colab cell.
    started_at = time.monotonic()
    last_heartbeat = started_at
    process = subprocess.Popen(command, cwd=str(cwd) if cwd else None, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    assert process.stdout is not None
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    selector = selectors.DefaultSelector()
    selector.register(process.stdout, selectors.EVENT_READ)
    log_file = open(log_path, 'w', encoding='utf-8') if log_path else None
    try:
        while selector.get_map():
            for key, _ in selector.select(timeout=1):
                chunk = os.read(key.fileobj.fileno(), 4096)
                if not chunk:
                    selector.unregister(key.fileobj)
                    continue
                output = decoder.decode(chunk)
                print(output, end='', flush=True)
                if log_file:
                    log_file.write(output)
                    log_file.flush()
            now = time.monotonic()
            if process.poll() is None and now - last_heartbeat >= 30:
                heartbeat = f'[Colab heartbeat] process is running | elapsed={(now - started_at) / 60:.1f}m\n'
                print(heartbeat, end='', flush=True)
                if log_file:
                    log_file.write(heartbeat)
                    log_file.flush()
                last_heartbeat = now
        remaining = decoder.decode(b'', final=True)
        if remaining:
            print(remaining, end='', flush=True)
            if log_file:
                log_file.write(remaining)
    finally:
        selector.close()
        if log_file:
            log_file.close()
    return_code = process.wait()
    if return_code != 0:
        log_hint = f' Full log: {log_path}' if log_path else ''
        raise RuntimeError(f'Command failed with exit code {return_code}.{log_hint}')

run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if (REPO_PATH / '.git').is_dir():
    run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_PATH)
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_PATH)
    run(['git', 'checkout', BRANCH], cwd=REPO_PATH)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_PATH)
else:
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_PATH])
run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH)

UV = shutil.which('uv') or 'uv'
run([UV, 'venv', '--clear', '--python', '3.10', '.venv'], cwd=REPO_PATH)
PYTHON = REPO_PATH / '.venv/bin/python'
run([UV, 'pip', 'install', '--python', PYTHON, 'torch==2.7.1', 'torchvision==0.22.1', '--index-url', 'https://download.pytorch.org/whl/cu126'])
run([UV, 'pip', 'install', '--python', PYTHON, '-r', 'requirements.txt'], cwd=REPO_PATH)
run([PYTHON, '-c', "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); assert torch.cuda.is_available(), 'Colab GPU is not enabled'; print('GPU:', torch.cuda.get_device_name(0))"])

override = f"import yaml; from pathlib import Path; p=Path('configs/hada_tcformer.yaml'); c=yaml.safe_load(p.read_text()); c['max_epochs_loso']=125; c['preprocessing']['physionet']['trial_duration']=4.1; c['preprocessing']['physionet']['batch_size']=48; c['preprocessing']['physionet']['accumulate_grad_batches']=2; c['preprocessing']['physionet']['riemannian_alignment']={USE_RA!r}; c['preprocessing']['physionet'].setdefault('model_overrides', {{}})['im_tta_steps']={IM_TTA_STEPS if USE_IM_TTA else 0}; p.write_text(yaml.safe_dump(c, sort_keys=False))"
run([PYTHON, '-c', override], cwd=REPO_PATH)

MNE_DATA.mkdir(parents=True, exist_ok=True)
TRAIN_LOG.parent.mkdir(parents=True, exist_ok=True)
environment = os.environ.copy()
environment.update({'PYTHONUNBUFFERED': '1', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True', 'MPLBACKEND': 'Agg', 'MNE_DATA': str(MNE_DATA), 'MNE_DATASETS_EEGBCI_PATH': str(MNE_DATA)})
print('===== TRAIN FULL-MAMBA HADA-TCFORMER | PHYSIONET S001-S020 LOSO | 4.1 s | MICRO-BATCH 48 x ACCUM 2 = EFFECTIVE BS 96 | 125 EPOCHS =====', flush=True)
print(f'Options: RA={USE_RA} | IM-TTA steps={IM_TTA_STEPS if USE_IM_TTA else 0}', flush=True)
print('Live log:', TRAIN_LOG, flush=True)
run([PYTHON, '-u', 'train_pipeline.py', '--model', 'hada_tcformer', '--dataset', 'physionet', '--loso', '--gpu_id', '0'], cwd=REPO_PATH, env=environment, stream=True, log_path=TRAIN_LOG)

RESULT_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
archive = shutil.make_archive(str(RESULT_ARCHIVE), 'zip', root_dir=REPO_PATH, base_dir='results')
print('Train complete. Results:', archive, flush=True)
